In [0]:
from pyspark.sql import functions as F

import os
current_dir = os.path.dirname(os.path.abspath("__file__"))
path = f"{current_dir}/eurostat_mean-median_2026.csv"
df_raw = spark.read.csv(path, header=True, inferSchema=True)

df_filtered = df_raw.filter(
    (F.col("TIME_PERIOD") == 2022) & 
    (F.col("nace_r2") == "B-S_X_O")
)

# Map country codes to Polish names
country_names_pl = {
    "AT": "Austria", "BE": "Belgia", "BG": "Bułgaria", "HR": "Chorwacja",
    "CY": "Cypr", "CZ": "Czechy", "DK": "Dania", "EE": "Estonia",
    "FI": "Finlandia", "FR": "Francja", "DE": "Niemcy", "EL": "Grecja",
    "HU": "Węgry", "IE": "Irlandia", "IT": "Włochy", "LV": "Łotwa",
    "LT": "Litwa", "LU": "Luksemburg", "MT": "Malta", "NL": "Holandia",
    "PL": "Polska", "PT": "Portugalia", "RO": "Rumunia", "SK": "Słowacja",
    "SI": "Słowenia", "ES": "Hiszpania", "SE": "Szwecja", "CH": "Szwajcaria",
    "AL": "Albania", "BA": "Bośnia i Hercegowina", "IS": "Islandia",
    "MK": "Macedonia Północna", "NO": "Norwegia", "RS": "Serbia"
}

# Map country codes to English names
country_names_en = {
    "AT": "Austria", "BE": "Belgium", "BG": "Bulgaria", "HR": "Croatia",
    "CY": "Cyprus", "CZ": "Czechia", "DK": "Denmark", "EE": "Estonia",
    "FI": "Finland", "FR": "France", "DE": "Germany", "EL": "Greece",
    "HU": "Hungary", "IE": "Ireland", "IT": "Italy", "LV": "Latvia",
    "LT": "Lithuania", "LU": "Luxembourg", "MT": "Malta", "NL": "Netherlands",
    "PL": "Poland", "PT": "Portugal", "RO": "Romania", "SK": "Slovakia",
    "SI": "Slovenia", "ES": "Spain", "SE": "Sweden", "CH": "Switzerland",
    "AL": "Albania", "BA": "Bosnia and Herzegovina", "IS": "Iceland",
    "MK": "North Macedonia", "NO": "Norway", "RS": "Serbia"
}

# Convert dictionaries to Spark mappings
country_mapping_pl = F.create_map([F.lit(x) for pair in country_names_pl.items() for x in pair])
country_mapping_en = F.create_map([F.lit(x) for pair in country_names_en.items() for x in pair])

# Pivot data
df_pivoted = df_filtered.groupBy(
    F.col("geo").alias("Kod_Kraju")
).pivot("indic_se", ["MEAN_E_EUR", "MED_E_EUR"]).agg(F.first("OBS_VALUE"))


df_final = df_pivoted.withColumnRenamed("MEAN_E_EUR", "Srednia") \
                     .withColumnRenamed("MED_E_EUR", "Mediana") \
                     .withColumn("Kraj", country_mapping_pl[F.col("Kod_Kraju")]) \
                     .withColumn("Country_EN", country_mapping_en[F.col("Kod_Kraju")]) \
                     .withColumn("Rozwarstwienie_Pct", F.round(((F.col("Srednia") - F.col("Mediana")) / F.col("Srednia")) * 100, 2)) \
                     .filter(F.col("Srednia").isNotNull() & F.col("Mediana").isNotNull()) \
                     .filter(~F.col("Kod_Kraju").isin(["EA19", "EA20", "EU27_2020"])) \
                     .orderBy(F.col("Rozwarstwienie_Pct").desc())

display(df_final)

Databricks visualization. Run in Databricks to view.